# Ch 6. 구조화 출력

원본: [AI Assistant Engineering - Part 2 Ch 6](https://desty.github.io/study-ai-assistant-engineering/part2/06-structured-output/)

## 이 챕터에서 배우는 것

- 자유 텍스트 응답이 **파이프라인의 적**인 이유
- **세 가지 구조화 출력 방법** — 프롬프트 힌트 / Tool-use 스키마 / Native JSON mode
- **Pydantic** 으로 검증 + 실패 시 **자동 재프롬프트**
- Nested · Optional · Enum · List 등 실전 스키마 패턴
- 큰따옴표 이스케이프 · 날짜·통화 포맷 · 필드 누락 — 자주 터지는 파싱 지옥

In [ ]:
# 필수 패키지
!pip install -q anthropic pydantic

In [ ]:
# API 키 설정
import os
from getpass import getpass

if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')

In [ ]:
from anthropic import Anthropic
from pydantic import BaseModel
import json

client = Anthropic()

## 4. 최소 예제 — Pydantic 모델로 추출

In [ ]:
class Order(BaseModel):
    item: str
    quantity: int
    address: str

SYSTEM = """주문 텍스트에서 정보를 추출해 **오직 JSON만** 반환하세요.
스키마: {"item": str, "quantity": int, "address": str}
JSON 외 다른 텍스트 금지."""

r = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=256,
    system=SYSTEM,
    messages=[{"role": "user", "content": "빨간 운동화 2켤레 서울 강남구로 보내주세요"}],
)

order = Order.model_validate_json(r.content[0].text)  # (1)!
print(f"아이템: {order.item}, 수량: {order.quantity}, 주소: {order.address}")

## 5. 실전 튜토리얼

### 5.2 Tool-use로 구조화 출력 (Anthropic 방식)

In [ ]:
tools = [{
    "name": "record_order",
    "description": "고객 주문을 기록한다",
    "input_schema": {  # (1)!
        "type": "object",
        "properties": {
            "item":     {"type": "string", "description": "제품명"},
            "quantity": {"type": "integer", "minimum": 1},
            "address":  {"type": "string"},
        },
        "required": ["item", "quantity", "address"],
    },
}]

r = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1024,
    tools=tools,
    tool_choice={"type": "tool", "name": "record_order"},  # (2)!
    messages=[{"role": "user", "content": "빨간 운동화 2켤레 서울 강남구로 보내주세요"}],
)

# tool_use 응답에서 input 을 꺼냄
for block in r.content:
    if block.type == "tool_use":
        data = block.input  # 이미 dict
        print(f"추출된 주문: {data}")

### 5.3 Pydantic 검증 + 자동 재프롬프트

In [ ]:
from pydantic import ValidationError

class Order(BaseModel):
    item: str
    quantity: int
    address: str

SYSTEM = """주문 정보를 JSON 으로만 반환하세요.
스키마: {"item": str, "quantity": int, "address": str}
다른 텍스트 절대 금지."""

def extract_order(text: str, retries: int = 2) -> Order | None:
    messages = [{"role": "user", "content": text}]
    last_error = None

    for attempt in range(retries + 1):
        r = client.messages.create(
            model="claude-haiku-4-5-20251001", max_tokens=256,
            system=SYSTEM, messages=messages,
        )
        raw = r.content[0].text.strip()

        # 1) JSON 추출: 중괄호로 감싸진 첫 덩어리만
        start = raw.find("{")
        end = raw.rfind("}") + 1
        if start < 0 or end <= 0:
            last_error = "JSON 블록을 찾을 수 없음"
        else:
            # 2) 파싱 + 검증
            try:
                return Order.model_validate_json(raw[start:end])
            except (json.JSONDecodeError, ValidationError) as e:
                last_error = str(e)

        # 3) 에러 메시지를 다음 프롬프트에 포함해 재시도
        if attempt < retries:
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content":
                f"이전 응답이 스키마를 어겼습니다: {last_error}\n다시 올바른 JSON만 반환하세요."})

    return None  # 최종 실패

# 테스트
result = extract_order("빨간 운동화 2켤레 서울 강남구로")
if result:
    print(f"성공: {result}")
else:
    print("실패")

### 5.4 실전 스키마 패턴

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from datetime import date

class Address(BaseModel):
    street: str
    city: str
    postal_code: str | None = None  # Optional

class Order(BaseModel):
    order_id: str = Field(..., pattern=r"^[A-Z]-\d+$")  # (1)!
    items: list[str]                                     # List
    quantity: int = Field(..., ge=1, le=100)             # 범위 제약
    priority: Literal["low", "normal", "high"]           # Enum
    ship_date: date                                      # ISO-8601 자동 파싱
    address: Address                                     # Nested
    notes: str | None = None                              # Optional

# JSON Schema 추출 예
schema = Order.model_json_schema()
print(json.dumps(schema, ensure_ascii=False, indent=2)[:500] + "...")

## 6. 자주 깨지는 포인트

### 실수 1. 큰따옴표 이스케이프 누락
- 모델이 텍스트 안에 큰따옴표가 있으면 잘못된 JSON 생성
- 대응: 시스템 프롬프트에 명시, 또는 tool-use 방식 사용

### 실수 2. 날짜·통화 포맷 혼란
- `"ship_date": "다음주 월요일"` 이나 `"price": "₩15,000"`
- 대응: 스키마에 예시값 명시, Pydantic Field description 활용

### 실수 3. 필수 필드 누락
- 모델이 가끔 어떤 필드를 빠뜨림
- 대응: Pydantic에서 `...` 명확히 Required 지정, 검증 실패 시 재프롬프트

### 실수 4. 스키마가 너무 깊고 복잡
- 5단계 이상 nested 스키마는 문제
- 대응: 스키마를 2~3단계로 flat하게, 복잡한 구조는 여러 번 호출로 쪼개기

### 실수 5. 재시도 무한 루프
- 검증 실패할 때마다 재질의 → 비용 폭주
- 대응: 재시도 상한 2~3회, 초과 시 fallback 경로

## 7. 운영 시 체크할 점

- [ ] **Pydantic 모델을 단일 모듈**에 집중 (`schemas.py`)
- [ ] **tool-use 방식 기본** — 프롬프트 힌트만 쓰는 구간이 있으면 마이그레이션 계획
- [ ] **검증 실패율** 로깅 — 5% 이상이면 스키마/프롬프트 재설계 신호
- [ ] **재시도 상한** 2~3회 + fallback 경로 필수
- [ ] 스키마 변경 시 **하위 호환** 고려
- [ ] LLM 응답 샘플을 **테스트 fixture** 로 저장 (회귀 테스트용)
- [ ] **민감 정보 필드** (주민번호·카드번호 등)는 스키마 레벨에서 거부

---

**다음 챕터** → Ch 7. 스트리밍과 UX

지금까지 **한 번에 전체 응답**을 받았습니다. 사용자 체감 속도를 올리는 **토큰 단위 스트리밍** 으로.